In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [4]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role":"system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

In [5]:
response.output[0].content[0].parsed

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [6]:
response.output[0].content[0].text

'{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

In [7]:
event = response.output_parsed

In [8]:
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [9]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent
)
print(response.output_text)

{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}


In [10]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Event: Science Fair  
Participants: Alice and Bob  
Day: Friday


## Structured RAG

In [11]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 385 chunks from 95 documents


In [12]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [13]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [14]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    return response.output_text

In [15]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [16]:
answer = rag('how do I implement LLM as a Judge?')

In [19]:
def llm_structured(
        user_prompt,
        output_type,
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed



In [20]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [21]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [22]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [ ]:
def rag_structured(query, output_type=REGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    